In [18]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import cv2
from torch.utils.data import Dataset, DataLoader

In [19]:
class CRNN(nn.Module):
    def __init__(self, num_classes, img_height=32):
        super(CRNN, self).__init__()
        self.img_height = img_height
        self.num_classes = num_classes

        self.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1)
        self.conv4 = nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv5 = nn.Conv2d(256, 512, kernel_size=3, stride=1, padding=1)
        self.bn5   = nn.BatchNorm2d(512)
        self.conv6 = nn.Conv2d(512, 512, kernel_size=3, stride=1, padding=1)
        self.bn6   = nn.BatchNorm2d(512)
        self.pool4 = nn.MaxPool2d(kernel_size=(1, 2), stride=2)

        self.conv7 = nn.Conv2d(512, 512, kernel_size=2, stride=1, padding=0)
        self.bn7   = nn.BatchNorm2d(512)
        self.pool5 = nn.MaxPool2d(kernel_size=(1, 2), stride=(1, 2))

        self.BiLSTM = nn.LSTM(512, 256, num_layers=2, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        x = F.relu(self.pool1(self.conv1(x)))
        x = F.relu(self.pool2(self.conv2(x)))
        x = F.relu(self.conv3(x))
        x = F.relu(self.pool3(self.conv4(x)))

        x = F.relu(self.bn5(self.conv5(x)))
        x = F.relu(self.bn6(self.conv6(x)))
        x = self.pool4(x)

        x = F.relu(self.bn7(self.conv7(x)))
        x = self.pool5(x)

        b, c, h, w = x.size()
        x = x.permute(0, 3, 1, 2)
        x = x.contiguous().view(b, w, c * h)
        x, _ = self.BiLSTM(x)
        x = self.fc(x)
        return x.log_softmax(dim=2)

In [21]:
def encode(text, label_to_index):
    return [label_to_index[c] for c in text]

In [22]:
def preprocess_image(image, img_height=32):
    if len(image.shape) == 3:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    h, w = image.shape
    new_w = max(1, int(w * img_height / h))
    image = cv2.resize(image, (new_w, img_height))
    image = image.astype(np.float32) / 255.0
    return image

In [23]:
labels = "0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"
label_to_index = {ch: idx + 1 for idx, ch in enumerate(labels)}
index_to_label = {v: k for k, v in label_to_index.items()}

In [24]:
criterion = nn.CTCLoss(blank=0, zero_infinity=True)

In [ ]:
def load_dataset(txt_path):
    samples = []
    with open(txt_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split(" ", 1)
            if len(parts) == 2:
                samples.append((parts[0], parts[1]))
    return samples

data = load_dataset("/kaggle/input/datasets/allahhitler/ocr-synthetic-dataset/labels.txt")

In [26]:
class OCRDataset(Dataset):
    def __init__(self, samples, img_dir, label_to_index, img_height=32):
        self.samples = samples
        self.img_dir = img_dir
        self.label_to_index = label_to_index
        self.img_height = img_height

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_name, label = self.samples[idx]
        img_path = os.path.join(self.img_dir, img_name)
        image = cv2.imread(img_path)
        image = preprocess_image(image, self.img_height)
        image = torch.FloatTensor(image).unsqueeze(0)
        encoded = encode(label, self.label_to_index)
        return image, torch.LongTensor(encoded)

In [ ]:
def collate_fn(batch):
    images, labels = zip(*batch)
    max_w = max(img.shape[2] for img in images)
    padded = torch.zeros(len(images), 1, images[0].shape[1], max_w)
    for i, img in enumerate(images):
        padded[i,:,:,:img.shape[2]] = img
    label_lengths = torch.LongTensor([len(l) for l in labels])
    labels_cat = torch.cat(list(labels))
    return padded, labels_cat, label_lengths


img_path = "/kaggle/input/datasets/allahhitler/ocr-synthetic-dataset/images"
batch_size = 32
epochs = 10
lr = 0.001

split = int(0.9 * len(data))
train_data, val_data = data[:split], data[split:]

train_loader = DataLoader(
    OCRDataset(train_data, img_path, label_to_index),
    batch_size=batch_size, shuffle=True,  collate_fn=collate_fn, num_workers=2
)
val_loader = DataLoader(
    OCRDataset(val_data, img_path, label_to_index),
    batch_size=batch_size, shuffle=False, collate_fn=collate_fn, num_workers=2
)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

NameError: name 'data' is not defined

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(label_to_index) + 1

model = CRNN(num_classes=num_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

In [ ]:
def greedy_decode(log_probs, index_to_label):
    pred_indices = log_probs.argmax(dim=1).tolist()
    decoded, prev = [], None
    for idx in pred_indices:
        if idx != prev and idx != 0:
            decoded.append(index_to_label.get(idx, ""))
        prev = idx
    return "".join(decoded)

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for images, labels, label_lengths in loader:
        images = images.to(device)
        labels = labels.to(device)
        label_lengths = label_lengths.to(device)

        outputs = model(images)
        outputs = outputs.permute(1, 0, 2)
        T = outputs.size(0)
        input_lengths = torch.full((images.size(0),), T, dtype=torch.long, device=device)

        loss = criterion(outputs, labels, input_lengths, label_lengths)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)


def val_epoch(model, loader, criterion, device, index_to_label):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels, label_lengths in loader:
            images = images.to(device)
            labels = labels.to(device)
            label_lengths = label_lengths.to(device)

            outputs = model(images)
            T = outputs.size(1)
            input_lengths = torch.full((images.size(0),), T, dtype=torch.long, device=device)

            loss = criterion(outputs.permute(1, 0, 2), labels, input_lengths, label_lengths)
            total_loss += loss.item()

            offset = 0
            for i in range(images.size(0)):
                pred = greedy_decode(outputs[i], index_to_label)
                gt_len = label_lengths[i].item()
                gt = "".join(index_to_label.get(idx, "") for idx in labels[offset:offset + gt_len].tolist())
                offset += gt_len
                correct += int(pred.lower() == gt.lower())
                total += 1

    return total_loss / len(loader), correct / total if total else 0.0

In [ ]:
best_val_loss = float("inf")

for epoch in range(1, epochs + 1):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = val_epoch(model, val_loader, criterion, device, index_to_label)

    print(f"Epoch {epoch}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "crnn_best.pth")